# Question 7 - Cost Tracking

```text
pytest full suite
      ↓
SQL agent calls ─────┐
                     │
Judge calls ─────────┤
                     ↓
               usage tracker
                     ↓
              token totals
                     ↓
             estimated cost
```

**Number of tests ≠ Number of LLM requests.** One SQL-agent run generally needs a request before schema inspection, another before SQL execution, and another for its structured answer. Question 5 adds a separate judge run.

`conftest.py` automatically resets and enables tracking at session start, prints the summary at session end, and removes the patch afterward. `patch_agent.py` wraps the `Agent.run` class method, covering both agents, and reads each new `ModelResponse`'s model name and usage. Only new responses are counted, avoiding prior-history double counting. Existing tests and SQL-agent behavior are unchanged.

The exact module `patch_agent.py` and `conftest.py` were not found locally. The price table and formula were found in `01-foundation/02-rag/openai.ipynb` and preserved: **gpt-4o-mini input $0.15 and output $0.60 per million tokens**. This is the course's standard-rate estimate; cached-input discounts are not applied.

Compatibility with installed PydanticAI 2.63: usage fields are `input_tokens` and `output_tokens` on each response's `usage`; no legacy `request_tokens`/`response_tokens` names are used. Caller-based agent-name inference is preserved through the wrapper using the installed `_infer_name` implementation. Provider snapshot names map to the course model alias for pricing.

The following is the captured output of the completed `uv run pytest test_agent.py -s` run, rather than an estimate based on eight tests. It records completed model responses; transport retries without a recorded response are not independently counted.

In [1]:
# Captured from the actual completed full-suite run.
measured_summary = '========================= LLM usage and estimated cost =========================\n\nPricing: course MODEL_PRICES; USD; standard input/output token rates.\nCached-input discounts are not applied (course approximation).\nCompleted agent runs: agent=8, judge=1\ngpt-4o-mini-2024-07-18: requests=25, input_tokens=10810, output_tokens=1119, total_tokens=11929, estimated_cost=$0.00229290\nTOTAL: requests=25, input_tokens=10810, output_tokens=1119, total_tokens=11929\nApproximate total cost: $0.00229290\n============================== 8 passed in 42.28s =============================='
print(measured_summary)

========================= LLM usage and estimated cost =========================

Pricing: course MODEL_PRICES; USD; standard input/output token rates.
Cached-input discounts are not applied (course approximation).
Completed agent runs: agent=8, judge=1
gpt-4o-mini-2024-07-18: requests=25, input_tokens=10810, output_tokens=1119, total_tokens=11929, estimated_cost=$0.00229290
TOTAL: requests=25, input_tokens=10810, output_tokens=1119, total_tokens=11929
Approximate total cost: $0.00229290
============================== 8 passed in 42.28s ==============================


In [2]:
import re
from decimal import Decimal

# Read the measured totals from the captured summary, not from test count.
match = re.search(
    r"TOTAL: requests=(\d+), input_tokens=(\d+), output_tokens=(\d+), total_tokens=(\d+)",
    measured_summary,
)
requests, input_tokens, output_tokens, total_tokens = map(int, match.groups())
assert total_tokens == input_tokens + output_tokens
estimated_cost = (
    Decimal(input_tokens) * Decimal("0.15")
    + Decimal(output_tokens) * Decimal("0.60")
) / Decimal(1_000_000)
recorded_cost = Decimal(re.search(r"Approximate total cost: \$(\d+\.\d+)", measured_summary).group(1))
assert estimated_cost == recorded_cost
print(f"Requests: {requests}")
print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {output_tokens}")
print(f"Total tokens: {total_tokens}")
print(f"Verified estimated cost: ${estimated_cost:.8f}")

Requests: 25
Input tokens: 10810
Output tokens: 1119
Total tokens: 11929
Verified estimated cost: $0.00229290


**Approximate total cost of the full test suite: $0.00229290.**

This is the cost of the final single full-suite run. A future run may have different token totals and cost.